# 🔗 文字接龙：用线性回归和逻辑回归预测下一个字

湖北理工学院《人工智能理论》课程资料

📝 **本节内容概述：**
1. 📊 **数据准备**：从预训练语料中抽取纯中文短句作为实验数据
2. ✂️ **分词**：用最朴素的逐字切分方式构建字符词表
3. 🔢 **嵌入**：将每个字映射为一个自然数编号
4. 🧠 **训练**：引入滑动窗口，分别用线性回归和逻辑回归训练"文字接龙"模型
5. 📈 **评估**：用混淆矩阵和 AUC 对比两种方法的优劣
6. 🚀 **部署**：用训练好的模型做自回归接龙，演示有无停止符的差异

> 💡 **核心理念**：本节实验的最终目的不是追求多高的准确率，而是让你理解——大语言模型（如 ChatGPT）在最底层的本质，就是**文字接龙**：给定前面的字，预测下一个字。而这个预测过程，本质上就是一个**数学问题**。我们用前两节学过的线性回归和逻辑回归来亲手实现它，打破对 LLM 的神秘感。

## 🎯 什么是文字接龙？

想象你在手机上打字，输入法会自动猜测你接下来要打的字——这就是最朴素的文字接龙。

**给定前面几个字，预测下一个字。** 就这么简单。

$$\text{"深度学习是"} \xrightarrow{\text{预测}} \text{"用"}$$

ChatGPT、文心一言等大语言模型（LLM），它们在最核心的推理环节做的事情，和上面的公式一模一样——只不过它们用了更深的网络、更多的数据、更大的算力。但核心逻辑不变：**猜下一个字（token）**。

---

### 我们的实验流程

| 步骤 | 名称 | 做什么 |
| :---: | :--- | :--- |
| 1️⃣ | **分词 (Tokenization)** | 把文本拆成最小单元（本实验中是单个汉字） |
| 2️⃣ | **嵌入 (Embedding)** | 给每个字分配一个数字编号，让计算机能做数学运算 |
| 3️⃣ | **训练 (Training)** | 用滑动窗口构造"根据前几个字猜下一个字"的样本，训练模型 |
| 4️⃣ | **评估 (Evaluation)** | 看看模型猜得准不准 |
| 5️⃣ | **部署 (Deployment)** | 让模型真正开始接龙！ |

## 📊 1. 数据准备

我们从本地语料文件 `./data/custom_pretrain_corpus.txt` 中加载文本数据。该文件包含了若干条关于人工智能、机器学习的中文短句。

为了让实验简洁聚焦，我们只抽取**纯中文句子**（不含英文字母和阿拉伯数字），确保所有训练样本都是干净的中文字符序列。

In [ ]:
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ========================================================
# 👇 设置中文字体，确保图表正常显示汉字
# ========================================================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========================================================
# 👇 加载语料并筛选纯中文句子
# ========================================================
with open('./data/custom_pretrain_corpus.txt', 'r', encoding='utf-8') as f:
    all_lines = [line.strip() for line in f if line.strip()]

print(f'📁 语料文件共 {len(all_lines)} 行')

# 筛选规则：只保留纯中文字符和中文标点的句子（排除含英文字母、数字的行）
def is_pure_chinese(text):
    return bool(re.fullmatch(r'[\u4e00-\u9fff\u3000-\u303f\uff00-\uffef，。！？、；：""\'\'（）《》【】]+', text))

chinese_lines = [line for line in all_lines if is_pure_chinese(line)]

# 去重：很多语料是模板生成的重复句式，保留唯一的句子
chinese_lines = list(dict.fromkeys(chinese_lines))

# 取前 50 条作为我们的迷你语料库
corpus = chinese_lines[:50]

print(f'✅ 筛选后共 {len(chinese_lines)} 条纯中文语料，取前 {len(corpus)} 条作为实验数据')
print(f'\n📜 前 10 条样例：')
for i, line in enumerate(corpus[:10]):
    print(f'  [{i}] {line}')

## ✂️ 2. 分词 (Tokenization)

计算机看不懂汉字，它只认识数字。所以，我们必须要把文本切分成一个个小的单元（Token），然后给每个单元编上号。

对于中文而言，最简单的分词方式就是**逐字切分**。例如："机器学习" $\rightarrow$ `['机', '器', '学', '习']`。

在本实验中，我们会构建一个**字符表 (Vocabulary)**，包含：
1. 语料库中出现的所有汉字、标点。
2. 两个特殊的占位符：
   - `<PAD>`: 用于填充，保证序列长度一致（后面训练时会用到）。
   - `<EOS>`: 结束符（End Of Sentence），标记一句话的完结。

In [ ]:
# 1. 统计所有字符
all_chars = "".join(corpus)
unique_chars = sorted(list(set(all_chars)))

# 2. 构建词表（加入特殊标记）
# 约定：<PAD> 为 0，<EOS> 为 1
vocab = ["<PAD>", "<EOS>"] + unique_chars

# 3. 构建映射字典：字 -> 编号，编号 -> 字
char2idx = {char: i for i, char in enumerate(vocab)}
idx2char = {i: char for i, char in enumerate(vocab)}

vocab_size = len(vocab)
print(f"📌 词表构建完成！词表大小 (Vocab Size): {vocab_size}")
print(f"🔑 前 20 个字符：{vocab[:20]}")
print(f"🔍 示例: '学' 的编号是 {char2idx['学']}, 编号 10 的字是 '习' ")

## 🔢 3. 嵌入 (Embedding)

有了词表之后，我们就可以把每一句中文转换成一串数字了。这种将离散的文字转换为数值向量的操作叫作**嵌入**。

虽然现代 LLM 使用高维连续向量（如 512 维、4096 维），但本实验为了直接衔接前两章的回归 and 分类，我们使用**最朴素的自然数编号**来表示每个字。这既可以作为回归模型的特征，也可以作为分类模型的类别标签。

In [ ]:
# ========================================================
# 👇 将语料库转换为数字序列
# ========================================================

def text_to_sequence(text, add_eos=True):
    seq = [char2idx[char] for char in text]
    if add_eos:
        seq.append(char2idx["<EOS>"])
    return seq

# 转换整个语料库
corpus_seqs = [text_to_sequence(line) for line in corpus]

# 展示转换效果
example_text = corpus[0]
example_seq = corpus_seqs[0]
print(f"📄 原始文本: {example_text}")
print(f"🔢 数字序列: {example_seq}")
print(f"📏 序列长度: {len(example_seq)} (含 <EOS>)")

## 🧠 4. 训练 (Training)

现在进入最有趣的部分：我们要教计算机如何"文字接龙"。

### 4.1 滑动窗口构建训练样本

在训练之前，我们需要把长长的句子拆解成一个个"根据前文猜下文"的样本。这里引入**滑动窗口 (Sliding Window)** 的概念：
- 设定窗口大小 `window_size` (比如 4)。
- 用前 4 个字作为**输入 (X)**，第 5 个字作为**标签 (y)**。
- 窗口向后滑动一位，循环往复。

例如句子："机器学习是让..."
1. `[机, 器, 学, 习]` $\rightarrow$ `是`
2. `[器, 学, 习, 是]` $\rightarrow$ `让`

In [ ]:
window_size = 4
X_data = []
y_data = []

for seq in corpus_seqs:
    if len(seq) <= window_size:
        continue
    for i in range(len(seq) - window_size):
        X_data.append(seq[i : i + window_size])
        y_data.append(seq[i + window_size])

X_np = np.array(X_data)
y_np = np.array(y_data)

print(f"📊 样本构建完成！总样本数: {len(X_np)}")
print(f"📂 输入 X 形状: {X_np.shape}, 标签 y 形状: {y_np.shape}")

# 展示一个具体的对应关系
print("\n🔍 映射示例（前 3 个样本）：")
for i in range(3):
    chars_x = "".join([idx2char[idx] for idx in X_np[i]])
    char_y = idx2char[y_np[i]]
    print(f"  [{chars_x}] -> [{char_y}] (数字: {X_np[i]} -> {y_np[i]})")

### 4.2 方法一：线性回归 (Linear Regression)

线性回归通常用来预测**连续值**（如房价）。如果强行用它来接龙，实际上是把"猜下一个字"看成了一个**数值预测**问题：猜下一个字的编号是多少。

**优化点**：为了演示更真实的训练过程，我们引入**小批量训练 (Mini-batch)** 和 **随机打乱 (Shuffle)**。这可以防止模型陷入局部最优，并提高泛化能力。

In [ ]:
X_t = torch.tensor(X_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.float32).view(-1, 1)

# 初始化参数：4 个输入维度（window_size），1 个输出维度
w_lin = torch.randn(window_size, 1, requires_grad=True)
b_lin = torch.zeros(1, requires_grad=True)

batch_size = 16
learning_rate = 0.001
epochs = 2000
loss_history_lin = []

num_samples = X_t.shape[0]

for epoch in range(epochs):
    # --- 1. 随机打乱样本顺序 ---
    indices = torch.randperm(num_samples)
    X_shuffled = X_t[indices]
    y_shuffled = y_t[indices]
    
    total_loss = 0
    # --- 2. 小批量训练 ---
    for i in range(0, num_samples, batch_size):
        X_batch = X_shuffled[i : i + batch_size]
        y_batch = y_shuffled[i : i + batch_size]
        
        # 前向传播
        y_pred = torch.mm(X_batch, w_lin) + b_lin
        loss = torch.mean((y_pred - y_batch) ** 2) / 2
        
        # 反向传播
        loss.backward()
        
        with torch.no_grad():
            w_lin -= learning_rate * w_lin.grad
            b_lin -= learning_rate * b_lin.grad
            w_lin.grad.zero_()
            b_lin.grad.zero_()
        
        total_loss += loss.item() * X_batch.size(0)
    
    avg_loss = total_loss / num_samples
    loss_history_lin.append(avg_loss)
    
    if (epoch + 1) % 200 == 0 or epoch == 0:
        clear_output(wait=True)
        plt.figure(figsize=(8, 4))
        plt.plot(loss_history_lin, color='dodgerblue')
        plt.title(f"线性回归训练损失 (Epoch {epoch+1}, Batch Size=16, Shuffled)")
        plt.xlabel("Epoch")
        plt.ylabel("MSE Loss")
        plt.grid(alpha=0.3)
        plt.show()

print(f"✅ 线性回归训练完成！最终平均 Loss: {loss_history_lin[-1]:.4f}")

### 4.3 方法二：逻辑回归（Softmax 多分类）

文字接龙本质上是一个从 243 个候选汉字中选出一个最可能的字的"分类"问题。这正是现代 LLM 的核心方案。

**同理**：我们在这里也采用 **随机打乱** 和 **小批量 (batch_size=16)** 进行训练。通过多轮迭代，模型会学习到哪些字经常出现在哪些字后面。

In [ ]:
# 标签转换：ID 分类问题需要 Long 类型索引
y_t_long = torch.tensor(y_np, dtype=torch.long)

# 初始化参数：4 个输入，词表大小 (vocab_size) 个输出概率点
W_clf = torch.randn(window_size, vocab_size, requires_grad=True)
b_clf = torch.zeros(vocab_size, requires_grad=True)

learning_rate = 0.5
epochs = 2000
loss_history_clf = []

for epoch in range(epochs):
    # --- 1. 随机打乱 ---
    indices = torch.randperm(num_samples)
    X_shuffled = X_t[indices]
    y_shuffled = y_t_long[indices]
    
    total_loss = 0
    # --- 2. 小批量训练 ---
    for i in range(0, num_samples, batch_size):
        X_batch = X_shuffled[i : i + batch_size]
        y_batch = y_shuffled[i : i + batch_size]
        
        # 前向传播：计算各类别得分 (Logits)
        logits = torch.mm(X_batch, W_clf) + b_clf
        
        # 交叉熵损失（内置 Softmax）
        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        
        # 反向传播
        loss.backward()
        
        with torch.no_grad():
            W_clf -= learning_rate * W_clf.grad
            b_clf -= learning_rate * b_clf.grad
            W_clf.grad.zero_()
            b_clf.grad.zero_()
            
        total_loss += loss.item() * X_batch.size(0)
        
    avg_loss = total_loss / num_samples
    loss_history_clf.append(avg_loss)
    
    if (epoch + 1) % 200 == 0 or epoch == 0:
        clear_output(wait=True)
        plt.figure(figsize=(8, 4))
        plt.plot(loss_history_clf, color='darkorange')
        plt.title(f"逻辑回归训练损失 (Epoch {epoch+1}, Batch Size=16, Shuffled)")
        plt.xlabel("Epoch")
        plt.ylabel("Cross Entropy Loss")
        plt.grid(alpha=0.3)
        plt.show()

print(f"✅ 逻辑回归训练完成！最终平均 Loss: {loss_history_clf[-1]:.4f}")

## 📈 5. 评估 (Evaluation)

训练完成后，我们对比两种模型的预测性能：

1. **准确率 (Accuracy)**：分别统计两个模型"判断对错"的能力。
2. **ROC 曲线**：衡量逻辑回归模型（分类）在不同置信度阈值下的整体分类能力。

> 💡 **注**：线性回归预测的是单一数值，没有"概率"的概念，因此通常无法绘制标准的 ROC 曲线。这也从侧面反映了分类模型（逻辑回归）在概率建模上的优越性。

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

with torch.no_grad():
    # --- 1. 获取预测结果 ---
    lin_outputs = torch.mm(X_t, w_lin) + b_lin
    lin_preds = torch.round(lin_outputs).clamp(0, vocab_size - 1).squeeze().long().numpy()
    
    clf_logits = torch.mm(X_t, W_clf) + b_clf
    clf_probs = torch.nn.functional.softmax(clf_logits, dim=1).numpy()
    clf_preds = np.argmax(clf_probs, axis=1)

    y_true = y_np
    labels = np.arange(vocab_size)

# --- 2. 准确率对比 ---
lin_acc = np.mean(lin_preds == y_true)
clf_acc = np.mean(clf_preds == y_true)

print(f"📊 【线性回归】 总准确率 Accuracy: {lin_acc:.4f}")
print(f"📊 【逻辑回归】 总准确率 Accuracy: {clf_acc:.4f}")

# --- 3. 可视化对比图 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# A: 准确率柱状图
ax1.bar(['线性回归', '逻辑回归'], [lin_acc, clf_acc], color=['dodgerblue', 'darkorange'], alpha=0.8)
ax1.set_ylim(0, 1.0)
ax1.set_title("预测准确率对比 (Match Accuracy)", fontsize=14)
ax1.set_ylabel("准确率 (Accuracy)")
for i, v in enumerate([lin_acc, clf_acc]):
    ax1.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')

# B: 逻辑回归的 ROC 曲线 (Macro-average)
y_true_bin = label_binarize(y_true, classes=labels)
n_classes = vocab_size

fpr = dict()
tpr = dict()
roc_auc = dict()

present_classes = np.unique(y_true)
all_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(all_fpr)

count = 0
for i in present_classes:
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], clf_probs[:, i])
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    count += 1

mean_tpr /= count
macro_auc = auc(all_fpr, mean_tpr)

ax2.plot(all_fpr, mean_tpr, color='darkorange', lw=2, label=f'Macro-average ROC (AUC = {macro_auc:.2f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('假正率 (False Positive Rate)')
ax2.set_ylabel('真正率 (True Positive Rate)')
ax2.set_title('逻辑回归：宏平均 ROC 曲线', fontsize=14)
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()

## 🚀 6. 部署 (Deployment) 

真正的接龙并不是只猜一个字，而是**自回归 (Autoregressive)** 生成：把刚猜出来的字放回输入末端，继续猜下一个字。

### 6.1 生成函数实现
在生成过程中，模型会不断预测下一个字的概率。

In [ ]:
def generate_text(start_text, model_W, model_b, max_len=20):
    current_seq = [char2idx[c] for c in start_text]
    result = list(start_text)
    
    for _ in range(max_len):
        # 输入最后的 window_size 个字
        input_x = torch.tensor([current_seq[-window_size:]], dtype=torch.float32)
        
        with torch.no_grad():
            logits = torch.mm(input_x, model_W) + model_b
            next_idx = torch.argmax(logits, dim=1).item()
            
        result.append(idx2char[next_idx])
        current_seq.append(next_idx)
        
    return "".join(result)

prompt = "神经网络"
print(f"💡 输入：{prompt}")

print(f"✍️ 输出：{generate_text(prompt, W_clf, b_clf)}")

## 🎓 课堂总结

通过本节实验，我们完成了一次从零开始的"大模型雏形"构建：

1. **数学本质**：将复杂的人类语言建模成了计算机能理解的**概率预测问题**。
2. **分词与嵌入**：汉字通过编码转换成了模型能够计算的数字序列。
3. **分类 vs 回归**：我们亲证了：对于文本这种高度抽象的离散符号，**分类（逻辑回归）** 模型比 **回归（线性回归）** 模型要适合得多。这解释了为什么现代 LLM 的输出层本质上都是巨大的 Softmax 分类器。

这就是 LLM 的"秘密"：它并不是真的在理解真理，而是在做一场规模宏大的数学接龙游戏。